<a href="https://colab.research.google.com/github/Grazipolachini/MBA/blob/main/ETL_RISCO_DE_CREDITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Parte 1 - Ingestão de dados na camada Raw**

In [1]:
#Instala Pyspark

!pip install pyspark

In [3]:
#Inicia motor de processamento distribuido

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EtlRiscoDecredito") \
    .getOrCreate()

spark

In [4]:
#Monta drike para ler arquivos
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
#Le arquivos
df_app = spark.read.csv(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/Raw/application_train.csv",
    header=True,
    inferSchema=True
)

In [6]:
#Le arquivos
df_bureau = spark.read.csv(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/Raw/bureau.csv",
    header=True,
    inferSchema=True
)

In [7]:
df_app.printSchema()
df_app.show(5)
df_app.count()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullable = true)
 |-- OWN_CAR_AG

307511

In [8]:
df_bureau.printSchema()
df_bureau.show(5)
df_bureau.count()

root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_CURRENCY: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)

+----------+------------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------

1716428

In [9]:
#Salva como arquivo parquet
df_app.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/raw_parquet/application_train")

df_bureau.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/EtlRiscoDeCredito/data/raw_parquet/bureau"
)

**Parte 2 - Tratamento na camada Trusted**

In [10]:
print("Linhas:", df_app.count())
print("Colunas:", len(df_app.columns))
print("Linhas:", df_bureau.count())
print("Colunas:", len(df_bureau.columns))

Linhas: 307511
Colunas: 122
Linhas: 1716428
Colunas: 17


In [11]:
#Valida distribuição da base
df_app.groupBy("TARGET").count().show()
#Valida duplicidade na chave
df_app.select("SK_ID_CURR").distinct().count()
df_app.count()

+------+------+
|TARGET| count|
+------+------+
|     1| 24825|
|     0|282686|
+------+------+



307511

In [12]:
#Validação de nulos na base

from pyspark.sql.functions import col, count, when

total = df_app.count()

null_df = df_app.select([
    (count(when(col(c).isNull(), c)) / total).alias(c)
    for c in df_app.columns
])

null_df.show()

+----------+------+------------------+-----------+------------+---------------+------------+----------------+----------+--------------------+--------------------+--------------------+----------------+-------------------+------------------+-----------------+--------------------------+----------+-------------+-----------------+---------------+------------------+----------+--------------+---------------+----------------+----------+----------+-------------------+--------------------+--------------------+---------------------------+--------------------------+-----------------------+--------------------------+--------------------------+---------------------------+----------------------+----------------------+-----------------------+-----------------+------------------+--------------------+-------------------+------------------+------------------+---------------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+--------

In [13]:
from pyspark.sql.functions import col, when

df_app_clean = df_app.withColumn(
    "DAYS_EMPLOYED",
    when(col("DAYS_EMPLOYED") == 365243, None)
    .otherwise(col("DAYS_EMPLOYED"))
)

In [14]:
df_app_clean.filter(col("DAYS_EMPLOYED") == 365243).count()

0

In [15]:
df_app_clean.filter(col("DAYS_EMPLOYED").isNull()).count()

55374

In [16]:
from pyspark.sql.functions import col

df_app_clean = df_app_clean.fillna(
    {"DAYS_EMPLOYED": 0}
)

In [ ]:
df_app_clean.filter(col("DAYS_EMPLOYED") == 0).count()

55376

In [19]:
from pyspark.sql.functions import col, when, sum as spark_sum
from pyspark.sql.types import IntegerType, LongType

# Pegar colunas int e bigint
int_columns = [
    field.name
    for field in df_app_clean.schema.fields
    if isinstance(field.dataType, (IntegerType, LongType))
]

# Preencher todas com 0
df_app_clean = df_app_clean.fillna(0, subset=int_columns)
int_with_nulls

[]

In [20]:
from pyspark.sql.functions import col, when, sum as spark_sum

# Contar nulos novamente
null_counts_after = df_app_clean.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in int_columns
]).collect()[0].asDict()

null_counts_after

{'SK_ID_CURR': 0,
 'TARGET': 0,
 'CNT_CHILDREN': 0,
 'DAYS_BIRTH': 0,
 'DAYS_EMPLOYED': 0,
 'DAYS_ID_PUBLISH': 0,
 'FLAG_MOBIL': 0,
 'FLAG_EMP_PHONE': 0,
 'FLAG_WORK_PHONE': 0,
 'FLAG_CONT_MOBILE': 0,
 'FLAG_PHONE': 0,
 'FLAG_EMAIL': 0,
 'REGION_RATING_CLIENT': 0,
 'REGION_RATING_CLIENT_W_CITY': 0,
 'HOUR_APPR_PROCESS_START': 0,
 'REG_REGION_NOT_LIVE_REGION': 0,
 'REG_REGION_NOT_WORK_REGION': 0,
 'LIVE_REGION_NOT_WORK_REGION': 0,
 'REG_CITY_NOT_LIVE_CITY': 0,
 'REG_CITY_NOT_WORK_CITY': 0,
 'LIVE_CITY_NOT_WORK_CITY': 0,
 'FLAG_DOCUMENT_2': 0,
 'FLAG_DOCUMENT_3': 0,
 'FLAG_DOCUMENT_4': 0,
 'FLAG_DOCUMENT_5': 0,
 'FLAG_DOCUMENT_6': 0,
 'FLAG_DOCUMENT_7': 0,
 'FLAG_DOCUMENT_8': 0,
 'FLAG_DOCUMENT_9': 0,
 'FLAG_DOCUMENT_10': 0,
 'FLAG_DOCUMENT_11': 0,
 'FLAG_DOCUMENT_12': 0,
 'FLAG_DOCUMENT_13': 0,
 'FLAG_DOCUMENT_14': 0,
 'FLAG_DOCUMENT_15': 0,
 'FLAG_DOCUMENT_16': 0,
 'FLAG_DOCUMENT_17': 0,
 'FLAG_DOCUMENT_18': 0,
 'FLAG_DOCUMENT_19': 0,
 'FLAG_DOCUMENT_20': 0,
 'FLAG_DOCUMENT_21': 0}